In [ ]:
# ==============================================================================
# ENTERPRISE P&L PIPELINE: GOLD LAYER (ANALYTICS & DIMENSIONAL MODELING)
# ==============================================================================
import uuid
import traceback
from datetime import datetime
from zoneinfo import ZoneInfo

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, coalesce, trim, lower, lit, from_utc_timestamp, substring, 
    split, row_number, when, concat, concat_ws, to_date, lead
)
from pyspark.sql.window import Window

# ------------------------------------------------------------------------------
# 1. PIPELINE CONFIGURATION & SPARK INITIALIZATION
# ------------------------------------------------------------------------------
class PipelineConfig:
    PIPELINE_NAME      = "NB_Corp_Finance_DataHub_ELT_Gold_Profitability" 
    ENVIRONMENT        = "Production"
    TZ                 = "America/New_York"
    RUN_ID             = str(uuid.uuid4())
    TARGET_CATALOG     = "LH_Corp_DataHub"
    
    # POWER BI SEMANTIC MODEL
    SEMANTIC_MODEL     = "SM_Corp_Project_Real_Profitability" 
    
    # Schema Definitions
    SCHEMA_SILVER      = "silver"
    SCHEMA_DATAVERSE   = "dataverse"         
    SCHEMA_DV_HISTORY  = "dataverse_history" 
    SCHEMA_HR          = "hr"                
    SCHEMA_GOLD        = "gold"

# Initialize Spark Session with strict schema and calendar configurations
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# [FIX] Handle legacy dates (pre-1582) found in UKG raw data to avoid write failures
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

# ------------------------------------------------------------------------------
# 2. TELEMETRY & UTILITIES
# ------------------------------------------------------------------------------
def log_info(msg):    print(f"{datetime.now(ZoneInfo(PipelineConfig.TZ)).strftime('%Y-%m-%d %H:%M:%S')} - [INFO] - {msg}")
def log_error(msg):   print(f"{datetime.now(ZoneInfo(PipelineConfig.TZ)).strftime('%Y-%m-%d %H:%M:%S')} - [ERROR] - {msg}")
def log_success(msg): print(f"{datetime.now(ZoneInfo(PipelineConfig.TZ)).strftime('%Y-%m-%d %H:%M:%S')} - [SUCCESS] - {msg}")

def add_metadata(df: DataFrame) -> DataFrame:
    return (df.withColumn("_gold_modified_at", from_utc_timestamp(F.current_timestamp(), PipelineConfig.TZ))
              .withColumn("_gold_pipeline_run_id", lit(PipelineConfig.RUN_ID)))

# ------------------------------------------------------------------------------
# 3. CORE TRANSFORMATION ENGINE (GOLD LAYER)
# ------------------------------------------------------------------------------
def build_gold_profitability_fact():
    """
    [Gold Transformation] Advanced Analytics Engine (Fact Table)
    - Implements Dynamic Dimension Routing (Umbrella Projects logic).
    - Computes Intercompany and Cross-Charge analytical flows based on Org hierarchy.
    - Includes 'org_key' to bridge cross-charge transactions in Power BI with dim_organization.
    """
    log_info("Reading Silver Fact Table (Source of Truth)...")
    df_silver = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_SILVER}.fact_time_and_cost_enriched")
    
    # --- DIMENSION 1: MASTER PROJECTS ---
    df_proj_mst = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_DV_HISTORY}.msdyn_project") \
        .select(
            col("msdyn_projectid").alias("dv_project_id"), 
            col("msdyn_subject").alias("mst_name"),
            col("msdyn_contractorganizationalunitidname").alias("project_company"),
            col("msdyn_comments"), 
            col("aux_customername").alias("mst_customer"),
            col("aux_industryname").alias("mst_industry"),
            col("aux_servicelinename").alias("mst_service_line"),
            col("aux_organizationname")
        ).withColumn("mst_L1", split(col("aux_organizationname"), "-").getItem(0)) \
         .withColumn("mst_L2", split(col("aux_organizationname"), "-").getItem(1)) \
         .withColumn("mst_L3", split(col("aux_organizationname"), "-").getItem(2)) \
         .drop("aux_organizationname") \
         .dropDuplicates(["dv_project_id"])

    # --- DIMENSION 2: FINANCIAL FALLBACK ---
    df_financials = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_DATAVERSE}.fact_financials_unified") \
        .withColumn("fin_proj_key", substring(col("project_name"), 1, 10))
    
    window_spec = Window.partitionBy("fin_proj_key").orderBy(col("transaction_date").desc())
    df_proj_fin = df_financials.withColumn("rn", row_number().over(window_spec)) \
        .filter(col("rn") == 1) \
        .select(
            col("fin_proj_key"),
            col("aux_customername").alias("fin_customer"),
            col("aux_industryname").alias("fin_industry"),
            col("aux_servicelinename").alias("fin_service_line"),
            col("reporting_org").alias("fin_company"),
            col("L1").alias("fin_L1"), col("L2").alias("fin_L2"), col("L3").alias("fin_L3")
        )

    # --- DIMENSION 3: TASK DIMENSIONS (UMBRELLA) ---
    df_task = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_DV_HISTORY}.msdyn_projecttask") \
        .select(
            col("msdyn_projecttaskid").alias("tid"), col("msdyn_subject").alias("task_name"),
            col("aux_customername").alias("tsk_customer"),
            col("aux_industryname").alias("tsk_industry"),
            col("aux_servicelinename").alias("tsk_service_line"),
            col("aux_organizationname")
        ).withColumn("tsk_L1", split(col("aux_organizationname"), "-").getItem(0)) \
         .withColumn("tsk_L2", split(col("aux_organizationname"), "-").getItem(1)) \
         .withColumn("tsk_L3", split(col("aux_organizationname"), "-").getItem(2)) \
         .drop("aux_organizationname") \
         .dropDuplicates(["tid"])

    # --- DIMENSION 4: RESOURCES ---
    df_res = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_DATAVERSE}.bookableresource_clean") \
        .select(col("bookableresourceid").alias("rid"), col("name").alias("dv_resource_name")).dropDuplicates(["rid"])

    # --- JOINING & PROJECTION ---
    df_stg1 = df_silver.alias("sil") \
        .join(df_proj_mst.alias("mst"), lower(trim(col("sil.projectid"))) == lower(trim(col("mst.dv_project_id"))), "left") \
        .withColumn("p_name", coalesce(col("mst.mst_name"), lit("Unmapped Project"))) \
        .withColumn("p_key", substring(col("p_name"), 1, 10))
        
    df_final = df_stg1.alias("stg") \
        .join(df_proj_fin.alias("fin"), col("stg.p_key") == col("fin.fin_proj_key"), "left") \
        .join(df_task.alias("tsk"), lower(trim(col("stg.projecttaskid"))) == lower(trim(col("tsk.tid"))), "left") \
        .join(df_res.alias("res"), lower(trim(col("stg.bookableresourceid"))) == lower(trim(col("res.rid"))), "left")

    df_resolved = df_final.select(
        col("stg.transaction_date"), col("stg.worked_hours"), col("stg.actual_cost_usd"),
        col("stg.synerion_emp_id"), col("stg.ukg_employee_id"),
        coalesce(col("res.dv_resource_name"), col("stg.employee_full_name")).alias("employee_full_name"),
        col("stg.employee_country"), col("stg.employee_reporting_category"),
        col("stg.L1").alias("employee_L1"), col("stg.L2").alias("employee_L2"), col("stg.L3").alias("employee_L3"),
        col("stg.p_key").alias("project_key"), col("stg.p_name").alias("project_name"),
        coalesce(col("stg.project_company"), lit("N/A")).alias("project_company"),
        
        # Waterfall dimension resolution for Umbrella Projects
        when(col("stg.msdyn_comments") == "Dimension by task", coalesce(col("tsk.tsk_customer"), lit("N/A"))) \
            .otherwise(coalesce(col("stg.mst_customer"), col("fin.fin_customer"), lit("N/A"))).alias("project_customer"),
        when(col("stg.msdyn_comments") == "Dimension by task", coalesce(col("tsk.tsk_industry"), lit("N/A"))) \
            .otherwise(coalesce(col("stg.mst_industry"), col("fin.fin_industry"), lit("N/A"))).alias("project_industry"),
        when(col("stg.msdyn_comments") == "Dimension by task", coalesce(col("tsk.tsk_service_line"), lit("N/A"))) \
            .otherwise(coalesce(col("stg.mst_service_line"), col("fin.fin_service_line"), lit("N/A"))).alias("project_service_line"),
        when(col("stg.msdyn_comments") == "Dimension by task", coalesce(col("tsk.tsk_L1"), lit("Corporate"))) \
            .otherwise(coalesce(col("stg.mst_L1"), col("fin.fin_L1"), lit("Corporate"))).alias("project_L1"),
        when(col("stg.msdyn_comments") == "Dimension by task", coalesce(col("tsk.tsk_L2"), lit("Corporate"))) \
            .otherwise(coalesce(col("stg.mst_L2"), col("fin.fin_L2"), lit("Corporate"))).alias("project_L2"),
        when(col("stg.msdyn_comments") == "Dimension by task", coalesce(col("tsk.tsk_L3"), lit("Corporate"))) \
            .otherwise(coalesce(col("stg.mst_L3"), col("fin.fin_L3"), lit("Corporate"))).alias("project_L3"),
        
        coalesce(col("tsk.task_name"), lit("Unmapped Task")).alias("task_name"),
        col("stg.subtask_category"),
        col("stg.pay_category_id"),
        col("stg.pay_category_description")
    )

    # --- ADVANCED ANALYTICS (FLOWS) ---
    df_analytics = df_resolved.withColumn(
        "project_country",
        when(col("project_company").contains("Costa Rica"), "Costa Rica")
        .when(col("project_company").contains("Colombia"), "Colombia")
        .otherwise("United States") 
    )
    
    emp_iso = when(col("employee_country") == "Costa Rica", "CR").when(col("employee_country") == "Colombia", "COL").when(col("employee_country") == "United States", "USA").otherwise("UNK")
    prj_iso = when(col("project_country") == "Costa Rica", "CR").when(col("project_country") == "Colombia", "COL").when(col("project_country") == "United States", "USA").otherwise("UNK")
    
    df_final_model = df_analytics.withColumn(
        "intercompany_flag", when(col("employee_country") == col("project_country"), "Local").otherwise("Intercompany")
    ).withColumn(
        "intercompany_flow", when(col("intercompany_flag") == "Local", concat(emp_iso, lit(" Local"))).otherwise(concat(emp_iso, lit(" -> "), prj_iso))
    ).withColumn(
        "cross_charge_flag", when(col("employee_L1") == col("project_L1"), "Direct").otherwise("Cross Charge")
    ).withColumn(
        "cross_charge_flow", when(col("cross_charge_flag") == "Direct", concat(col("employee_L1"), lit(" Direct"))).otherwise(concat(col("employee_L1"), lit(" -> "), col("project_L1")))
    ).withColumn(
        # Key to the conformed organization dimension (dim_organization): project L1|L2|L3.
        "org_key", F.concat_ws("|", col("project_L1"), col("project_L2"), col("project_L3"))
    )

    return df_final_model

def build_gold_dim_employee(df_fact_profitability: DataFrame):
    """
    [Gold Transformation] Static Employee Dimension (SCD Type 1)
    - Captures Master Employee details, dynamically handling LATAM custom fields.
    - Resolves missing US employee names by leveraging the Dataverse integration 
      already present in the Profitability Fact Table.
    - Sanitizes legacy effective dates to ensure Power BI semantic model stability.
    """
    log_info("Building Employee Dimension from UKG Master & Resolving Missing Names...")
    df_ukg = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_HR}.ukg_employment_details")
    
    # 1. Extract resolved names (Dataverse-enriched) from the Profitability Fact Table
    df_resolved_names = df_fact_profitability.select(
        col("ukg_employee_id").alias("fact_ukg_id"),
        col("employee_full_name").alias("resolved_name")
    ).filter(col("fact_ukg_id").isNotNull() & col("resolved_name").isNotNull()) \
     .dropDuplicates(["fact_ukg_id"])
    
    # 2. Join the UKG master dataset with the resolved names lookup
    df_joined = df_ukg.join(df_resolved_names, df_ukg.employeeid == df_resolved_names.fact_ukg_id, "left")
    
    return df_joined.select(
        col("employeeid").alias("ukg_employee_id"),
        col("employeenumber").alias("employee_number"),
        
        # Smart Name Resolution: 
        # Prioritizes the Dataverse-enriched name. If null, falls back to LATAM custom fields in UKG.
        coalesce(
            col("resolved_name"),
            when(
                col("p_config_name").isNotNull() & (trim(col("p_config_name")) != ""), 
                concat_ws(" ", col("p_config_name"), col("p_config_lastname1"))
            )
        ).alias("employee_full_name"),
        
        col("companyname").alias("legal_entity"),
        
        # Date Sanitization: Replaces invalid legacy dates (pre-1900) to prevent Power BI refresh errors
        when(to_date(col("lasthiredate")) < "1900-01-01", to_date(lit("1900-01-01")))
            .otherwise(to_date(col("lasthiredate"))).alias("hire_date"),
            
        when(to_date(col("terminationdate")) < "1900-01-01", to_date(lit("1900-01-01")))
            .otherwise(to_date(col("terminationdate"))).alias("termination_date"),
            
        col("employeestatuscode").alias("status_code")
    ).dropDuplicates(["ukg_employee_id"])

def build_gold_dim_employee_history():
    """
    [Gold Transformation] Employee History Dimension (SCD Type 2)
    - Maintains organizational hierarchy shifts over time.
    - [FIX]: Sanitizes ancient effective dates to protect Power BI engine.
    """
    log_info("Building Employee History SCD2 from UKG...")
    df_history = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_HR}.ukg_employee_job_history")
    df_dim_lookup = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_HR}.ukg_dimensions_details")
    
    # 1. Parse the job effective date and sanitize legacy values
    df_scd2 = df_history.withColumn(
        "raw_date", coalesce(col("jobEffectiveDate").cast("date"), to_date(col("jobEffectiveDate"), "yyyy-MM-dd HH:mm:ss"))
    ).withColumn(
        # Placeholder dates before 1900 (e.g. 0001-01-01) are set to 1900-01-01
        "effective_date", 
        when(col("raw_date") < "1900-01-01", to_date(lit("1900-01-01"))).otherwise(col("raw_date"))
    ).withColumn(
        "end_date", 
        coalesce(lead("effective_date").over(Window.partitionBy("employeeId").orderBy("effective_date")), to_date(lit("2099-12-31"), "yyyy-MM-dd"))
    )
    
    dim_l1 = df_dim_lookup.filter(col("level") == 1).select(col("code").alias("c1"), col("description").alias("d1"))
    
    return df_scd2.alias("hist") \
        .join(dim_l1, lower(trim(col("hist.orgLevel1Code"))) == lower(trim(col("c1"))), "left") \
        .withColumn("L1", split(col("d1"), "-").getItem(0)) \
        .select(
            col("employeeId").alias("ukg_employee_id"),
            col("L1"),
            col("effective_date"),
            col("end_date")
        ).filter(col("ukg_employee_id").isNotNull())

def build_gold_refresh_info():
    """
    [Gold Transformation] Pipeline Execution Metadata
    - LastRefresh: completion timestamp for front-end reporting visuals.
    - LastRevenueDate / LastSynerionDate (Fabric app, open item #33): global last T&M actual date and last
      Synerion hours date. This table has no relationship to dim_project, so row-level security never filters it.
    """
    log_info("Generating Last Refresh Date metadata...")
    gold = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}"
    return spark.sql(f"""
        SELECT
            from_utc_timestamp(current_timestamp(), '{PipelineConfig.TZ}') AS LastRefresh,
            (SELECT MAX(transaction_date) FROM {gold}.fact_revenue_metrics
              WHERE scenario = 'Actual' AND revenue_type = 'T&M')          AS LastRevenueDate,
            (SELECT MAX(transaction_date) FROM {gold}.fact_profitability_metrics) AS LastSynerionDate
    """)

def build_gold_dim_project(df_fact_profitability: DataFrame) -> DataFrame:
    """
    [Gold Transformation] Pure Project Dimension
    - Contains ONLY project-level attributes (ID, Name, Customer).
    - Organizational hierarchies (L1/L2/L3) have been moved to fact tables and a conformed org dimension
      to support multi-department billing (cross-charges) without granularity loss.
    """
    print("Building Pure Project Dimension (No Hierarchies)...")

    # 1. Operational Projects
    df_ops = df_fact_profitability.select(
        col("project_key"),
        col("project_name").alias("ops_name"),
        col("project_customer").alias("ops_customer")
    ).dropDuplicates(["project_key"]).filter(col("project_key").isNotNull())

    # 2. Financial Projects (SSOT)
    df_fin = spark.table("LH_Corp_DataHub.dataverse.fact_financials_unified").select(
        col("project_key"),
        col("Project_Name").alias("fin_name"),            
        col("aux_customername").alias("fin_customer")
    ).dropDuplicates(["project_key"]).filter(col("project_key").isNotNull())

    # 3. Full Outer Join
    df_joined = df_fin.join(df_ops, on="project_key", how="full_outer")

    # 4. Final Clean Projection: one row per key as the semantic model sees it. The model's relationships ignore
    #    case, so keys that differ only in case or spaces must be one row (open item #1, PROMETHEUS / Prometheus:
    #    two Project Ops projects, 2026-11-16). Facts keep their own key and org; prefer the row with a financial
    #    name, then the first key in binary order, so the pick is stable between runs.
    w_key = Window.partitionBy(F.upper(F.trim(F.col("project_key")))).orderBy(F.col("fin_name").isNull().asc(), F.col("project_key").asc())
    df_dim_final = (
        df_joined
        .withColumn("_rn", F.row_number().over(w_key))
        .filter(F.col("_rn") == 1)
        .select(
            F.col("project_key"),
            F.coalesce(F.col("fin_name"), F.col("ops_name"), F.col("project_key")).alias("project_name"),
            F.coalesce(F.col("fin_customer"), F.col("ops_customer"), F.lit("Unallocated Customer")).alias("project_customer"),
        )
    )

    return df_dim_final


# ------------------------------------------------------------------------------
# PROJECT MANAGER PER PROJECT AND PM APP ACCESS (2026-11-16)
# Rules: docs/04_decisions_and_open_items.md (#29, #30) and pipeline/security/README.md in the app repo.
# ------------------------------------------------------------------------------
# Manual access list for people outside the supervisor chain (executives, directors of other areas),
# uploaded to the default Lakehouse Files, Finance folder. Columns: user_upn, scope (all | area), area (L1, L1-L2 or L1-L2-L3).
PM_ACCESS_LIST_PATH = "Files/Finance/pm_performance_hub/pm_app_access_list.csv"
# The longest supervisor chain above a PM was 5 levels on 2026-11-16; this limit only stops a broken chain.
PM_ACCESS_MAX_CHAIN_LEVELS = 15
PM_ACCESS_EMPTY_SCHEMA = "user_upn string, project_key string, reason string"


def _auxis_id(c):
    """Auxis ID as UKG writes it: numeric numbers padded to 4 digits, letter prefixes (C, U, T...) kept."""
    c = trim(c.cast("string"))
    return when(c.rlike("^[0-9]+$"), F.lpad(c, 4, "0")).otherwise(c)


def _org_path(c):
    """Organization path for matching: upper case, no spaces (e.g. 'BPO-BTR-CORE')."""
    return F.upper(F.regexp_replace(coalesce(c, lit("")), r"\s+", ""))


def build_pm_per_project_key() -> DataFrame:
    """
    [Gold Helper] One Project Ops project per project key (open item #30)
    - Latest version of each msdyn_project record (SinkModifiedOn), deleted records excluded.
    - project_key is the first 10 characters of the project name (#11), so a key can hold several projects:
      the active one wins (statecode 0), then the most recently created.
    Returns key_norm, pm_user_id (Dynamics user id), pm_name and project_org (Project Ops organization path).
    """
    df_proj = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_DV_HISTORY}.msdyn_project")
    df_proj = (
        df_proj.withColumn("_v", row_number().over(Window.partitionBy("Id").orderBy(col("SinkModifiedOn").desc())))
        .filter(col("_v") == 1)
        .filter(col("IsDelete").isNull() | (col("IsDelete") == False))  # noqa: E712 (Spark column comparison)
        .withColumn("key_norm", F.upper(trim(substring(col("msdyn_subject"), 1, 10))))
    )
    w_key = Window.partitionBy("key_norm").orderBy(col("statecode").asc(), col("createdon").desc())
    return (
        df_proj.withColumn("_k", row_number().over(w_key))
        .filter(col("_k") == 1)
        .select(
            col("key_norm"),
            lower(trim(col("msdyn_projectmanager"))).alias("pm_user_id"),
            trim(col("msdyn_projectmanagername")).alias("pm_name"),
            _org_path(col("aux_organizationname")).alias("project_org"),
        )
    )


def add_project_manager(df_dim: DataFrame) -> DataFrame:
    """[Gold Transformation] dim_project + project_manager_name (open item #4). Keys without a Project Ops project stay blank."""
    df_pm = build_pm_per_project_key().select(col("key_norm"), col("pm_name").alias("project_manager_name"))
    return df_dim.withColumn("key_norm", F.upper(trim(col("project_key")))).join(df_pm, "key_norm", "left").drop("key_norm")


def build_gold_security_project_access(df_dim: DataFrame) -> DataFrame:
    """
    [Gold Transformation] PM app access: one row per (user_upn, project_key)
    - pm:        the project manager sees their projects.
    - manager:   everyone up the PM's supervisor chain (hr.vw_dim_employee_details) sees the PM's projects.
    - list_all:  people in the manual list with scope 'all' see every project.
    - list_area: people in the manual list with scope 'area' see the projects whose organization equals the area or
                 starts with the area followed by '-' (L1, L1-L2 or L1-L2-L3, whole segments). A project's
                 organizations are its Project Ops organization, its org in the hours (project_L1-L2-L3) and its
                 billing org (fact_financials_unified L1-L2-L3), as the app's project list shows them (Ion,
                 2026-11-20 and 2026-11-21: 'ITO' matched only 11 of 98 ITO projects).
    reason holds every rule that applies, comma-separated. The semantic model's dynamic role filters dim_project with it.
    Only Auxis ID, UPN and supervisor are read from the HR view; it also holds personal data the app must not use.
    """
    log_info("Building PM app access (PM, supervisor chain, manual list)...")
    df_keys = df_dim.select(col("project_key"), F.upper(trim(col("project_key"))).alias("key_norm")).distinct()
    df_pm = build_pm_per_project_key()
    df_resources = (
        spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_DATAVERSE}.bookableresource_clean")
        .filter(col("userid").isNotNull())
        .select(lower(trim(col("userid"))).alias("pm_user_id"), _auxis_id(col("aux_resourceno")).alias("emp"))
        .dropDuplicates(["pm_user_id"])
    )
    df_hr = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_HR}.vw_dim_employee_details").select(
        trim(col("employee_number").cast("string")).alias("emp"),
        trim(col("supervisor_employee_number").cast("string")).alias("sup"),
        lower(trim(col("user_principal_name"))).alias("upn"),
    )
    # project_key, key_norm, pm_user_id, pm_name, project_org, emp (the PM's Auxis ID)
    df_proj = df_keys.join(df_pm, "key_norm", "left").join(df_resources, "pm_user_id", "left")

    # 1. Project manager
    df_pm_rows = df_proj.join(df_hr.select("emp", "upn"), "emp").select(
        col("upn").alias("user_upn"), col("project_key"), lit("pm").alias("reason")
    )

    # 2. Supervisor chain, walked up one level per iteration
    df_edges = df_hr.filter("sup is not null and sup <> '' and sup <> emp").select("emp", "sup")
    df_frontier = df_proj.filter(col("emp").isNotNull()).select(col("project_key"), col("emp").alias("cur"))
    df_chain = None
    for _ in range(PM_ACCESS_MAX_CHAIN_LEVELS):
        df_frontier = (
            df_frontier.join(df_edges, df_frontier.cur == df_edges.emp)
            .select(col("project_key"), col("sup").alias("cur"))
            .distinct()
        )
        if df_frontier.limit(1).count() == 0:
            break
        df_chain = df_frontier if df_chain is None else df_chain.unionByName(df_frontier)
    if df_chain is None:
        df_mgr_rows = spark.createDataFrame([], PM_ACCESS_EMPTY_SCHEMA)
    else:
        df_mgr_rows = df_chain.join(df_hr.select(col("emp").alias("cur"), "upn"), "cur").select(
            col("upn").alias("user_upn"), col("project_key"), lit("manager").alias("reason")
        )

    # 3. Manual access list
    try:
        df_list = spark.read.option("header", True).csv(PM_ACCESS_LIST_PATH).select(
            lower(trim(col("user_upn"))).alias("user_upn"),
            lower(trim(col("scope"))).alias("scope"),
            _org_path(col("area")).alias("area"),
        )
    except Exception as e:
        log_info(f"PM app access list not found at {PM_ACCESS_LIST_PATH} ({type(e).__name__}): PM and supervisor-chain rows only.")
        df_list = spark.createDataFrame([], "user_upn string, scope string, area string")
    area_match = (col("project_org") == col("area")) | col("project_org").startswith(concat(col("area"), lit("-")))
    df_all_rows = df_list.filter(col("scope") == "all").crossJoin(df_keys.select("project_key")).select(
        col("user_upn"), col("project_key"), lit("list_all").alias("reason")
    )

    # Organizations per project for the area rule. Only adds projects: every project matched before by its Project Ops
    # organization still matches.
    # Hours org (Ion, 2026-11-20): the project's org in the hours (project_L1-L2-L3, what the app's L1/L2/L3 filter
    # shows). Customer projects often carry another organization in Project Ops, so 'ITO' matched only 11 ITO projects.
    df_hours_org = (
        spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.fact_profitability_metrics")
        .select(
            col("project_key"),
            _org_path(concat_ws("-", col("project_L1"), col("project_L2"), col("project_L3"))).alias("project_org"),
        )
        .join(df_keys.select("project_key"), "project_key")
        .filter(col("project_org") != "")
        .distinct()
    )
    # Billing org (Ion, 2026-11-21): projects billed in the area count too, as in the app's project list (hours org or
    # billing org). Same lines as build_gold_revenue_fact (Actual and Backlog, no licenses), read from the source because
    # this step runs before the revenue fact is written. NOR-401-06 (billed BPO/FAO, no hours yet) was missing for a BPO
    # area; for ITO, 13 projects and $17.0 M of Backlog.
    df_billing_org = (
        spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_DATAVERSE}.fact_financials_unified")
        .filter(col("scenario").isin("Actual", "Backlog") & (col("L3") != "LICS"))
        .select(col("project_key"), _org_path(concat_ws("-", col("L1"), col("L2"), col("L3"))).alias("project_org"))
        .join(df_keys.select("project_key"), "project_key")
        .filter(col("project_org") != "")
        .distinct()
    )
    df_orgs = (
        df_proj.filter(col("project_org") != "").select("project_key", "project_org")
        .unionByName(df_hours_org)
        .unionByName(df_billing_org)
        .distinct()
    )
    df_area_rows = (
        df_list.filter((col("scope") == "area") & (col("area") != ""))
        .join(df_orgs, area_match)
        .select(col("user_upn"), col("project_key"), lit("list_area").alias("reason"))
        .distinct()
    )

    df_access = (
        df_pm_rows.unionByName(df_mgr_rows).unionByName(df_all_rows).unionByName(df_area_rows)
        .filter(col("user_upn").isNotNull() & (col("user_upn") != ""))
        .groupBy("user_upn", "project_key")
        .agg(concat_ws(",", F.array_sort(F.collect_set("reason"))).alias("reason"))
    )

    # Data quality checks (counts and area codes only, no personal data)
    df_areas = df_list.filter(col("scope") == "area").select("area").distinct()
    df_matched = df_areas.join(df_orgs.select("project_org").distinct(), area_match, "left_semi")
    log_info(
        f"PM app access: {df_access.count()} rows, {df_access.select('user_upn').distinct().count()} users, "
        f"{df_access.select('project_key').distinct().count()} of {df_keys.count()} project keys covered."
    )
    log_info(f"Access list rows with a scope other than all/area: {df_list.filter(~col('scope').isin('all', 'area')).count()}")
    log_info(f"Access list areas that match no project: {[r.area for r in df_areas.join(df_matched, 'area', 'left_anti').collect()]}")
    log_info(
        "Access list UPNs not in the HR view: "
        f"{df_list.join(df_hr.select(col('upn').alias('user_upn')), 'user_upn', 'left_anti').select('user_upn').distinct().count()}"
    )
    log_info(
        "Project keys whose PM has no UPN: "
        f"{df_proj.filter(col('emp').isNotNull()).join(df_hr.filter(col('upn').isNotNull()).select('emp'), 'emp', 'left_anti').select('project_key').distinct().count()}"
    )
    return df_access

def build_gold_app_user_profile() -> DataFrame:
    """
    [Gold Transformation] App user profile: one row per UPN, for the PM app header (2026-11-16)
    - display_name: the Dataverse resource name (first name + last name, as in Project Operations), else the HR name.
    - job_title: UKG job catalog title for the employee's job code (hr.ukg_jobs_details.title via job_code);
      covers 2,547 of 2,548 people with a UPN on 2026-11-16.
    - One row per UPN: the active employee record first, then the highest Auxis ID.
    The semantic model filters it to the viewer's own row; only name and title are read, no other personal data.
    """
    log_info("Building app user profile (name and job title per UPN)...")
    df_hr = (
        spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_HR}.vw_dim_employee_details")
        .select(
            _auxis_id(col("employee_number")).alias("emp"),
            lower(trim(col("user_principal_name"))).alias("user_upn"),
            trim(col("employee_full_name")).alias("hr_name"),
            trim(col("job_code")).alias("job_code"),
            col("termination_date"),
        )
        .filter(col("user_upn").isNotNull() & (col("user_upn") != ""))
    )
    # Job catalog: one title per job code (active codes first; the catalog may repeat a code per country)
    w_job = Window.partitionBy("job_code").orderBy(col("isactive").desc_nulls_last(), col("countrycode").asc_nulls_last())
    df_jobs = (
        spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_HR}.ukg_jobs_details")
        .select(trim(col("jobcode")).alias("job_code"), trim(col("title")).alias("job_title"), col("isactive"), col("countrycode"))
        .filter(col("job_title").isNotNull() & (col("job_title") != ""))
        .withColumn("_rn", row_number().over(w_job))
        .filter(col("_rn") == 1)
        .select("job_code", "job_title")
    )
    df_names = (
        spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_DATAVERSE}.bookableresource_clean")
        .filter(col("aux_resourceno").isNotNull())
        .select(_auxis_id(col("aux_resourceno")).alias("emp"), trim(col("name")).alias("dv_name"))
        .dropDuplicates(["emp"])
    )
    w_upn = Window.partitionBy("user_upn").orderBy(col("termination_date").isNotNull().asc(), col("emp").desc())
    df_profile = (
        df_hr.join(df_jobs, "job_code", "left").join(df_names, "emp", "left")
        .withColumn("_rn", row_number().over(w_upn))
        .filter(col("_rn") == 1)
        .select(col("user_upn"), coalesce(col("dv_name"), col("hr_name")).alias("display_name"), col("job_title"))
    )
    log_info(
        f"App user profile: {df_profile.count()} users, "
        f"{df_profile.filter(col('job_title').isNotNull()).count()} with a job title."
    )
    return df_profile


def build_resource_candidates(df_rev: DataFrame) -> DataFrame:
    """
    [Gold Helper] Project Ops resource name -> Auxis ID (employee_number), one row per revenue line.
    - Lookup: dataverse.bookableresource_clean by normalized name. aux_resourceno lost its leading zeros in
      Dataverse ('82' vs UKG '0082'): numeric ids are padded to 4 digits; ids with a letter prefix stay as they are.
    - Homonyms (70 names shared by different people) are resolved, in order, by hire / termination dates on the
      line's transaction date, then by who logged Synerion hours on the project. Validated 2026-11-12: no fan-out,
      no line left ambiguous.
    Expects df_rev with columns _row_id, _name_key, transaction_date, project_key.
    """
    resource_no = trim(col("aux_resourceno"))
    df_lookup = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_DATAVERSE}.bookableresource_clean").select(
        lower(trim(F.regexp_replace(col("name"), r"\s+", " "))).alias("res_name_key"),
        when(resource_no.rlike("^[0-9]+$"), F.lpad(resource_no, 4, "0")).otherwise(resource_no).alias("res_employee_number"),
        to_date(col("aux_lasthiredate")).alias("res_hire_date"),
        to_date(col("aux_terminationdate")).alias("res_termination_date"),
    ).filter(
        col("res_name_key").isNotNull() & (col("res_name_key") != "") &
        col("res_employee_number").isNotNull() & (col("res_employee_number") != "")
    ).dropDuplicates(["res_name_key", "res_employee_number"])

    # Who logged Synerion hours on each project (synerion_emp_id is the Auxis ID)
    df_syn = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.fact_profitability_metrics") \
        .select(col("project_key").alias("syn_project_key"), col("synerion_emp_id").alias("syn_employee_number")) \
        .distinct()

    df_cand = df_rev.select("_row_id", "_name_key", "transaction_date", "project_key") \
        .join(df_lookup, col("_name_key") == col("res_name_key"), "left") \
        .join(df_syn, (col("project_key") == col("syn_project_key")) &
                      (col("res_employee_number") == col("syn_employee_number")), "left")

    emp = col("res_employee_number")
    valid_date = emp.isNotNull() & \
        (col("res_hire_date").isNull() | (col("res_hire_date") <= col("transaction_date"))) & \
        (col("res_termination_date").isNull() | (col("res_termination_date") >= col("transaction_date")))
    logged = col("syn_employee_number").isNotNull()

    return df_cand.groupBy("_row_id").agg(
        F.count(emp).alias("n_cand"),
        F.sum(when(valid_date, 1).otherwise(0)).alias("n_date"),
        F.sum(when(valid_date & logged, 1).otherwise(0)).alias("n_date_syn"),
        F.sum(when(logged, 1).otherwise(0)).alias("n_syn"),
        F.max(emp).alias("e_any"),
        F.max(when(valid_date, emp)).alias("e_date"),
        F.max(when(valid_date & logged, emp)).alias("e_date_syn"),
        F.max(when(logged, emp)).alias("e_syn"),
    )

def build_gold_revenue_fact():
    """
    [Gold Transformation] Revenue Fact Table
    - Extracts Actuals and Backlog records from Dataverse financials.
    - Excludes 'LICS' (Licenses) from L3 hierarchy to isolate services revenue.
    - Granularity: Project-Level Revenue for Financial Margin Analysis.
    - Generates 'org_key' to bridge cross-charge transactions via the Conformed Organization Dimension.
    - [NEW 2026-11-12] resource_name / resource_role from the Project Ops line, employee_number (Auxis ID) and
      resource_match, one row per input line:
        'unique'                 name belongs to one person
        'by_dates'               homonyms; exactly one was employed on the transaction date
        'by_dates_and_synerion'  several employed; exactly one of them logged Synerion hours on the project
        'by_synerion'            none by dates; exactly one logged Synerion hours on the project
        'ambiguous'              could not tell which person (employee_number blank)
        'role'                   planned at role level (resource_name = resource_role)
        'not_a_person'           name matches no resource (catalog services; see open item #27)
        'no_resource'            blank resource_name
    """
    log_info("Building Revenue Fact Table (Actuals & Backlog)...")
    df_fin = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_DATAVERSE}.fact_financials_unified")

    # 1. Filter out Licenses and include both Actuals and Backlog scenarios
    df_rev = df_fin.filter(
        (col("scenario").isin("Actual", "Backlog")) &
        (col("L3") != "LICS")
    ).select(
        to_date(col("transaction_date")).alias("transaction_date"),
        col("project_key"),

        # Financial Metrics
        col("amount").cast("double").alias("actual_revenue_usd"),
        col("hours_quantity").cast("double").alias("hours_quantity"),
        col("scenario"),
        col("type").alias("revenue_type"),

        # Organizational & Client Hierarchies
        col("L1").alias("financial_L1"),
        col("L2").alias("financial_L2"),
        col("L3").alias("financial_L3"),
        col("aux_customername").alias("financial_customer"),

        # Resource and role on the Project Ops line. Trimmed: names carry stray spaces (open item #20).
        trim(col("resource_name")).alias("resource_name"),
        trim(col("resource_role")).alias("resource_role"),

        # Key to the conformed organization dimension (dim_organization): financial L1|L2|L3.
        F.concat_ws("|", col("L1"), col("L2"), col("L3")).alias("org_key")
    ).withColumn("_row_id", F.monotonically_increasing_id()) \
     .withColumn("_name_key", lower(trim(F.regexp_replace(col("resource_name"), r"\s+", " "))))

    # 2. Person per line (Auxis ID), without fanning out
    df_stats = build_resource_candidates(df_rev)
    is_role = col("resource_name") == col("resource_role")
    df_out = df_rev.join(df_stats, "_row_id", "left").withColumn(
        "resource_match",
        when(col("resource_name").isNull() | (col("resource_name") == ""), lit("no_resource"))
        .when(is_role, lit("role"))
        .when(col("n_cand") == 0, lit("not_a_person"))
        .when(col("n_cand") == 1, lit("unique"))
        .when(col("n_date") == 1, lit("by_dates"))
        .when(col("n_date_syn") == 1, lit("by_dates_and_synerion"))
        .when(col("n_syn") == 1, lit("by_synerion"))
        .otherwise(lit("ambiguous"))
    ).withColumn(
        "employee_number",
        when(col("resource_match") == "unique", col("e_any"))
        .when(col("resource_match") == "by_dates", col("e_date"))
        .when(col("resource_match") == "by_dates_and_synerion", col("e_date_syn"))
        .when(col("resource_match") == "by_synerion", col("e_syn"))
    )

    return df_out.select(
        "transaction_date", "project_key", "actual_revenue_usd", "hours_quantity", "scenario", "revenue_type",
        "financial_L1", "financial_L2", "financial_L3", "financial_customer",
        "resource_name", "resource_role", "employee_number", "resource_match",
        "org_key"
    )

def build_gold_dim_organization() -> DataFrame:
    """
    [Gold Transformation] Conformed Organization Dimension
    - Extracts unique L1, L2, L3 combinations from both financial and operational fact tables.
    - Creates a composite 'org_key' to bridge cross-charge transactions in Power BI.
    """
    print("Building Organization Dimension (L1, L2, L3)...")
    
    # 1. Distinct project L1/L2/L3 combinations from the operational fact (Synerion hours)
    df_ops_org = spark.table("LH_Corp_DataHub.gold.fact_profitability_metrics").select(
        col("project_L1").alias("L1"),
        col("project_L2").alias("L2"),
        col("project_L3").alias("L3")
    )
    
    # 2. Distinct L1/L2/L3 combinations from the financial fact (revenue)
    df_fin_org = spark.table("LH_Corp_DataHub.dataverse.fact_financials_unified").select(
        col("L1"),
        col("L2"),
        col("L3")
    )
    
    # 3. Union both sources and remove duplicates
    df_org_union = df_ops_org.union(df_fin_org).dropDuplicates().fillna("Unallocated")
    
    # 4. Composite key L1|L2|L3, matched by the facts' org_key in the semantic model
    df_dim_org = df_org_union.withColumn(
        "org_key", 
        F.concat_ws("|", col("L1"), col("L2"), col("L3"))
    )
    
    return df_dim_org.select("org_key", "L1", "L2", "L3")

def build_gold_dim_hierarchy_master():
    """
    [Gold Transformation] Master Organizational Hierarchy Dimension
    - Combines unique structural hierarchies (L1, L2, L3) from both 
      Operational (Profitability) and Financial (Revenue) fact tables.
    - Filters data from '2026-01-01' onwards to ensure historical cleanliness.
    - Prevents blind spots caused by cross-charged resources or shared projects.
    """
    log_info("Building Unified Master Hierarchy Dimension (Profitability + Revenue)...")
    
    # 1. Extract distinct hierarchy from Operational Metrics (Synerion/UKG)
    df_profitability = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.fact_profitability_metrics") \
                            .filter(col("transaction_date") >= "2026-01-01") \
                            .select(
                                col("employee_L1").alias("L1"), 
                                col("employee_L2").alias("L2"), 
                                col("employee_L3").alias("L3")
                            )
    
    # 2. Extract distinct hierarchy from Financial Metrics (Revenue)
    # Revenue is filtered on transaction_date and mapped from financial_L1/L2/L3
    df_revenue = spark.table(f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.fact_revenue_metrics") \
                      .filter(col("transaction_date") >= "2026-01-01") \
                      .select(
                          col("financial_L1").alias("L1"), 
                          col("financial_L2").alias("L2"), 
                          col("financial_L3").alias("L3")
                      )
    
    # 3. Union both sources to capture every single active combination
    df_combined = df_profitability.union(df_revenue)
    
    # 4. Clean up nulls, empty strings, and eliminate duplicates
    df_hierarchy_final = df_combined.filter(col("L1").isNotNull() & (trim(col("L1")) != "")) \
                                    .dropDuplicates(["L1", "L2", "L3"])
    
    return df_hierarchy_final

# ------------------------------------------------------------------------------
# 4. AUTOMATION & EXECUTION
# ------------------------------------------------------------------------------
def refresh_powerbi_model(dataset_name: str):
    try:
        import sempy.fabric as fabric
        log_info(f"Triggering refresh for: '{dataset_name}'...")
        fabric.refresh_dataset(dataset=dataset_name)
        log_success("Semantic Model refresh triggered.")
    except Exception as e:
        log_error(f"PBI Refresh failed: {str(e)[:200]}")

if __name__ == "__main__":
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}")
        
        # 1. Profitability fact (hours and cost)
        df_fact = build_gold_profitability_fact()
        target_fact = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.fact_profitability_metrics"
        log_info(f"Writing to: {target_fact}...")
        df_fact.transform(add_metadata).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_fact)
        spark.sql(f"OPTIMIZE {target_fact}")
        
        # 2. Static Employee Dimension Sync (Enhanced with Fact Table dependency for name resolution)
        df_dim = build_gold_dim_employee(df_fact) 
        target_dim = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.dim_employee"
        
        log_info(f"Writing to: {target_dim}...")
        
        # Write the dataframe as a Delta table, overwriting the existing schema if necessary
        df_dim.transform(add_metadata) \
              .write \
              .format("delta") \
              .mode("overwrite") \
              .option("overwriteSchema", "true") \
              .saveAsTable(target_dim)
        
        # Optimize the Delta table for faster read performance in Direct Lake mode
        spark.sql(f"OPTIMIZE {target_dim}")

        # 3. Employee History (SCD2) Sync
        df_hist = build_gold_dim_employee_history()
        target_hist = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.dim_employee_history"
        log_info(f"Writing to: {target_hist}...")
        df_hist.transform(add_metadata).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_hist)
        spark.sql(f"OPTIMIZE {target_hist}")

        # 4. Project dimension (with project_manager_name, open item #4)
        df_dim_proj = add_project_manager(build_gold_dim_project(df_fact))
        target_dim_proj = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.dim_project"
        log_info(f"Writing to: {target_dim_proj}...")
        df_dim_proj.transform(add_metadata).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_dim_proj)
        spark.sql(f"OPTIMIZE {target_dim_proj}")

        # 4b. PM app access (row-level security source for the PM app, open items #29 and #30)
        df_access = build_gold_security_project_access(df_dim_proj)
        target_access = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.security_project_access"
        log_info(f"Writing to: {target_access}...")
        df_access.transform(add_metadata).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_access)
        spark.sql(f"OPTIMIZE {target_access}")

        # 4c. App user profile (name and job title in the PM app header)
        df_profile = build_gold_app_user_profile()
        target_profile = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.app_user_profile"
        log_info(f"Writing to: {target_profile}...")
        df_profile.transform(add_metadata).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_profile)
        spark.sql(f"OPTIMIZE {target_profile}")
        
        # 5. Revenue Fact Table Sync
        df_rev = build_gold_revenue_fact()
        target_rev = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.fact_revenue_metrics"
        log_info(f"Writing to: {target_rev}...")
        df_rev.transform(add_metadata).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_rev)
        spark.sql(f"OPTIMIZE {target_rev}")
        
        # 6. Refresh Metadata Sync
        df_refresh = build_gold_refresh_info()
        target_refresh = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.info_gold_profitability_refresh"
        log_info(f"Writing to: {target_refresh}...")
        df_refresh.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_refresh)
        
        # 7. Generate the master department hierarchy list for cross-functional reporting
        df_hierarchy = build_gold_dim_hierarchy_master()
        target_hierarchy_master = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.dim_Hierarchy_Master"
        
        log_info(f"Writing to: {target_hierarchy_master}...")
        
        # Write the dataframe as a Delta table, overwriting the existing schema if necessary
        df_hierarchy.transform(add_metadata) \
                    .write \
                    .format("delta") \
                    .mode("overwrite") \
                    .option("overwriteSchema", "true") \
                    .saveAsTable(target_hierarchy_master)
        
        # Optimize the Delta table for faster read performance in Direct Lake mode
        spark.sql(f"OPTIMIZE {target_hierarchy_master}")

        # 7.5 Conformed organization dimension (with org_key)
        df_organization = build_gold_dim_organization()
        target_org = f"{PipelineConfig.TARGET_CATALOG}.{PipelineConfig.SCHEMA_GOLD}.dim_organization"
        
        log_info(f"Writing to: {target_org}...")
        
        # Write as a Delta table
        df_organization.transform(add_metadata) \
                       .write \
                       .format("delta") \
                       .mode("overwrite") \
                       .option("overwriteSchema", "true") \
                       .saveAsTable(target_org)
        
        # Optimize the Delta table for Direct Lake reads
        spark.sql(f"OPTIMIZE {target_org}")

        # 8. Semantic Model Refresh
        refresh_powerbi_model(PipelineConfig.SEMANTIC_MODEL)
        log_success("Gold Pipeline execution complete.")
        
    except Exception as ex:
        log_error(f"Critical Failure: {str(ex)[:500]}")
        raise